# Comparación de modelos
modelos a comparar: CNN, LSTM, RNN, DNN y DAE. Además se debera emplear uno de los métodos de limpieza por transformada de Fourier/Wavelet como variable del feature engineering.

* agregar parametros de kurtosis, mean, min, max adicionales a los tensores
* agregar colab a un repo

| MODELO       | DESCRIPCION | FUNCION DE PERDIDA | OPTIMIZADOR | # PARAMETROS  | PRECISIÓN |
|--------------|--------------|-------------- |--------------|--------------|--------------|
| CNN          |        Col 2 |         Negative Log Likelihood |ADAM              | 175,256 | 87,76%|
| LSTM         |        Col 2 |         Col 3 |--------------|--------------|--------------|
| RNN          |        Col 2 |         Col 3 |--------------|--------------|--------------|
| DNN          |        Col 2 |         Col 3 |--------------|--------------|--------------|
| DAE          |        Col 2 |         Col 3 |--------------|--------------|--------------|


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import random_split, DataLoader, TensorDataset
import os
import pandas as pd

# Limpieza del dataset

El conjunto de datos IMS BEARING contiene metadatos que indican cuales de los 4 rodamientos presentaron fallas al final de las pruebas. Las 3 pruebas están compuestas de cierta cantidad de archivos que contienen las mediciones de los sensores en tiempos determinados. Sin embargo dichos archivos **no presentan etiquetas acerca de las fallas**. Hay investigadores como Miltos-90 quien publicó el repositorio [Failure_Classification_of_Bearings](https://github.com/Miltos-90/Failure_Classification_of_Bearings) donde realizó un análisis de los datos asignando a ciertos archivos una de cuatro etiquetas (early, suspect, normal, suspect, imminent failure). Estos archivos etiquetados representan intervalos dentro del conjunto de datos.


Para esta prueba se definirá simplemente el cuantil 85 de los datos para señalar aquellas mediciones que presentaban una falla en el rodamiento. Siendo entonces para el 2nd_test que se compone de 984 archivos, los primeros 836 etiquetados con 0 (para representar que no hay fallas) y los restantes 148 archivos etiquetados con 1. A los últimos 148 se les descartará los últimos 2 archivos dado que durante el EDA se pudo observar que estos solo presentaban ruido.



In [2]:
# Descarga del dataset
try:
    import gdown
except ImportError:
    !pip install gdown

# https://drive.google.com/file/d/10gqsuR_mHLCniKr2CTtAc1iPNeiNkOdD/view?usp=sharing
file_id = "10gqsuR_mHLCniKr2CTtAc1iPNeiNkOdD"
!gdown --id {file_id}

!unzip /content/NASA_IMS_bearing_dataset.zip >> /dev/null
!rm /content/NASA_IMS_bearing_dataset.zip



/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=10gqsuR_mHLCniKr2CTtAc1iPNeiNkOdD
From (redirected): https://drive.google.com/uc?id=10gqsuR_mHLCniKr2CTtAc1iPNeiNkOdD&confirm=t&uuid=7b4dcdd7-b91c-4d82-82b6-d3b2ebed7b58
To: /content/NASA_IMS_bearing_dataset.zip
100% 1.67G/1.67G [00:22<00:00, 73.0MB/s]


# Entrenamiento de una CNN

Se tomó el modelo descrito en el paper [Enhanced Fault Detection in Bearings Using Machine Learning and Raw Accelerometer Data: A Case Study Using the Case Western Reserve University Dataset](https://www.notion.so/Resumen-Enhanced-Fault-Detection-in-Bearings-Using-Machine-Learning-and-Raw-Accelerometer-Data-A-C-22b03b0ab1d2802ebefaf061da9b5429?source=copy_link) el cual se revisó previamente encontrando que obtuvieron resultados favorables en sus pruebas.


### Definición del conjunto de datos

In [3]:
# Lectura de cada archivo y creacion del conjunto de tensores
directory_path = '/content/2nd_test/2nd_test'
files = [f for f in os.listdir(directory_path) if os.path.isfile(os.path.join(directory_path, f))]

## IMPORTANTE: Se hara la suposición de que el 85% de los primeros
## archivos corresponden al estado normal de los rodamientos
umbral = int( len(os.listdir(directory_path)) * 0.85)
X_segments = []
y_segments = []
count = 0

# se remueven los 2 ultimos ya que estos presentan un ruido
for file_name in files[:-2]:
    file_path = os.path.join(directory_path, file_name)

    if not os.path.exists(file_path):
      pass

    # leer el archivo
    df = pd.read_csv(file_path, sep='\t', header=None) #dataframe con 20480 filas con canales
    data_tensor = torch.from_numpy(df.values).float() # Convert to float tensor. tensor de forma (20480, 4)
    reshaped_tensor = data_tensor.transpose(0, 1) # Transpose to get shape (4, 20480)
    segment = reshaped_tensor.unsqueeze(0) # aagregar una dimension para obtener la forma (1, 4, 20480)
    X_segments.append(segment)

    # validar si pasamos el umbral para asignar una clase
    count+=1
    if count > umbral:
      y_segments.append(torch.tensor(0))
    else:
      y_segments.append(torch.tensor(1))

X_train_tensor = torch.cat(X_segments, dim=0) # Shape: (num_segments, 1, segment_length)
y_train_tensor = torch.stack(y_segments) # Shape: (num_segments,)

# Create a TensorDataset and DataLoader
# train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Dataset completo
full_dataset = TensorDataset(X_train_tensor, y_train_tensor)

# Porcentaje de validación (por ejemplo, 20%)
val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size

# Dividir en train y val
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Definir DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

### Definición del modelo

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F

class Conv1DModel(nn.Module):
    def __init__(self, num_classes, input_channels=4, input_length=20480): # Added input_length
        super(Conv1DModel, self).__init__()

        # 1Fd — Conv1D, 64 Filtros, kernel 100, activación ReLU
        self.conv1 = nn.Conv1d(in_channels=input_channels, out_channels=64, kernel_size=100)

        # 2FD — Conv1D, 32 Filtros, kernel 50, activación ReLU
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=32, kernel_size=50)

        # 1PL — MaxPooling1D con tamaño 4
        self.pool = nn.MaxPool1d(kernel_size=4)

        # Calculate the size of the flattened output after convolutional and pooling layers
        # Pass a dummy tensor through the layers
        with torch.no_grad():
            dummy_input = torch.randn(1, input_channels, input_length) # Batch size 1, channels, length
            x = F.relu(self.conv1(dummy_input))
            x = F.relu(self.conv2(x))
            x = self.pool(x)
            flattened_size = torch.flatten(x, start_dim=1).shape[1]


        # 1FC — Capa Densa con 100 neuronas
        self.fc1 = nn.Linear(in_features=flattened_size, out_features=100)  # Use the calculated size

        # Capa de salida: tantas neuronas como clases
        self.fc_out = nn.Linear(100, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))    # Conv1 + ReLU
        x = F.relu(self.conv2(x))    # Conv2 + ReLU
        x = self.pool(x)             # MaxPooling

        x = torch.flatten(x, start_dim=1)  # Flatten

        x = F.relu(self.fc1(x))      # Fully connected + ReLU
        x = self.fc_out(x)           # Capa de salida

        return F.log_softmax(x, dim=1)

### Entrenamiento del modelo

In [15]:
# Implementación del modelo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu");
model = Conv1DModel(num_classes=2, input_channels=4, input_length=20480).to(device);
model

Conv1DModel(
  (conv1): Conv1d(4, 64, kernel_size=(100,), stride=(1,))
  (conv2): Conv1d(64, 32, kernel_size=(50,), stride=(1,))
  (pool): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=162656, out_features=100, bias=True)
  (fc_out): Linear(in_features=100, out_features=2, bias=True)
)

In [12]:

# Define loss function and optimizer
criterion = nn.NLLLoss() # Negative Log Likelihood Loss, suitable for log_softmax output
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 30 # Example number of epochs

for epoch in range(num_epochs):
    model.train() # Set the model to training mode
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader)}")

print("Training finished.")

Epoch 1/30, Loss: 0.7873110020160675
Epoch 2/30, Loss: 0.6352284336090088
Epoch 3/30, Loss: 0.5899137544631958
Epoch 4/30, Loss: 0.5411662101745606
Epoch 5/30, Loss: 0.5015421879291534
Epoch 6/30, Loss: 0.4674743163585663
Epoch 7/30, Loss: 0.45181755900382997
Epoch 8/30, Loss: 0.44046969294548033
Epoch 9/30, Loss: 0.44119101524353027
Epoch 10/30, Loss: 0.4316742867231369
Epoch 11/30, Loss: 0.4324567276239395
Epoch 12/30, Loss: 0.43069435834884645
Epoch 13/30, Loss: 0.4370262348651886
Epoch 14/30, Loss: 0.43229665398597716
Epoch 15/30, Loss: 0.42721827507019045
Epoch 16/30, Loss: 0.4305307298898697
Epoch 17/30, Loss: 0.4304568076133728
Epoch 18/30, Loss: 0.43244235873222353
Epoch 19/30, Loss: 0.4304949885606766
Epoch 20/30, Loss: 0.43059286415576936
Epoch 21/30, Loss: 0.4320564079284668
Epoch 22/30, Loss: 0.43372487723827363
Epoch 23/30, Loss: 0.4304776430130005
Epoch 24/30, Loss: 0.43539504408836366
Epoch 25/30, Loss: 0.43373316824436187
Epoch 26/30, Loss: 0.43046475410461427
Epoch 27/

### Evaluacion del modelo

In [13]:
# Evaluate the model on the validation set
model.eval() # Set the model to evaluation mode
running_loss = 0.0
correct = 0
total = 0

with torch.no_grad(): # Disable gradient calculation
    for inputs, labels in val_loader:
        # Move data to the same device as the model
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        running_loss += loss.item()

        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

val_loss = running_loss / len(val_loader)
val_accuracy = 100 * correct / total

print(f'Validation Loss: {val_loss:.4f}')
print(f'Validation Accuracy: {val_accuracy:.2f}%')


# validar otras metricas

Validation Loss: 0.3510
Validation Accuracy: 87.76%
